# Code 11.4.1: Loading HH-RLHF and checking a length bias

Loads the helpful subset of HH-RLHF, splits each record into the shared prompt and the two final responses, and measures how often the chosen response is longer. The prompt ends at the last `"\n\nAssistant:"`, where the two transcripts diverge. A few dozen records do not share a prompt under this rule and are skipped. On the training split, this prints about 43,800 usable pairs, with the chosen response longer (in characters) in about 59% of them.


In [ ]:
from datasets import load_dataset

ds = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base")

def split_prompt(chosen, rejected):
    """Both transcripts share the conversation so far; they differ only in the last assistant turn."""
    marker = "\n\nAssistant:"
    cut = chosen.rfind(marker) + len(marker)
    prompt = chosen[:cut]
    if not rejected.startswith(prompt):
        raise ValueError("chosen and rejected do not share the prompt")
    return prompt, chosen[cut:], rejected[cut:]

ex = ds["train"][0]
prompt, y_c, y_r = split_prompt(ex["chosen"], ex["rejected"])
print(prompt[-200:], "\nCHOSEN:", y_c[:200], "\nREJECTED:", y_r[:200])

n = longer = skipped = 0
for ex in ds["train"]:
    try:
        _, y_c, y_r = split_prompt(ex["chosen"], ex["rejected"])
    except ValueError:
        skipped += 1
        continue
    n += 1
    longer += len(y_c) > len(y_r)
print(f"pairs: {n}, skipped: {skipped}, chosen is longer: {longer / n:.3f}")
